# Tratamiento de Valores Faltantes — Ames Housing Dataset

## Objetivo de este notebook

Identificar, clasificar y tratar los **19 columnas con valores faltantes** del dataset.

La estrategia se basa en distinguir dos tipos de nulos con tratamientos distintos:
- **Nulos estructurales** → ausencia real de una característica (la propiedad no tiene piscina, garaje, etc.)
- **Nulos reales** → información que debería existir pero no fue registrada

> **Decisión clave:** Los nulos estructurales NO se imputan — se codifican como `"None"` o `0`
> para representar la ausencia real. Imputarlos con la media/moda introduciría información falsa.

In [ ]:
# ============================================================
# IMPORTACIÓN DE LIBRERÍAS
# ============================================================
import pandas as pd          # Manipulación y análisis tabular de datos
import numpy as np           # Operaciones numéricas
import matplotlib.pyplot as plt  # Visualización
import seaborn as sns        # Visualización estadística de alto nivel
from pathlib import Path     # Manejo de rutas de archivo multiplataforma

# Crea la carpeta images/ en la raíz del proyecto si no existe
Path('../../images').mkdir(exist_ok=True)

## 2. Carga del Dataset Original

Se carga `train.csv` sin `keep_default_na=False` para que pandas detecte los valores `"NA"`
del CSV de Kaggle como nulos reales — en esta fase el objetivo es identificar y analizar esos valores.

> **Nota:** A partir del notebook 03, se usa `keep_default_na=False, na_values=['']` para evitar
> que los `"None"` estructurales codificados en el CSV limpio sean interpretados como NaN.

In [3]:
data_path = Path("../../data/train.csv")
df = pd.read_csv(data_path)
print(df)

        Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0        1          60       RL         65.0     8450   Pave   NaN      Reg   
1        2          20       RL         80.0     9600   Pave   NaN      Reg   
2        3          60       RL         68.0    11250   Pave   NaN      IR1   
3        4          70       RL         60.0     9550   Pave   NaN      IR1   
4        5          60       RL         84.0    14260   Pave   NaN      IR1   
...    ...         ...      ...          ...      ...    ...   ...      ...   
1455  1456          60       RL         62.0     7917   Pave   NaN      Reg   
1456  1457          20       RL         85.0    13175   Pave   NaN      Reg   
1457  1458          70       RL         66.0     9042   Pave   NaN      Reg   
1458  1459          20       RL         68.0     9717   Pave   NaN      Reg   
1459  1460          20       RL         75.0     9937   Pave   NaN      Reg   

     LandContour Utilities  ... PoolArea PoolQC  Fe

## 3. Diagnóstico de Valores Faltantes

Se cuantifican y ordenan las columnas con nulos para priorizar el tratamiento.

Las columnas con mayor cantidad de nulos (`PoolQC`, `MiscFeature`, `Alley`) no representan
pérdida de información — la mayoría de propiedades simplemente **no tienen** esas características.

Se identificaron **19 columnas** con valores faltantes. Las cuatro más críticas por volumen son
`PoolQC` (1,453), `MiscFeature` (1,406), `Alley` (1,369) y `Fence` (1,179) — todas representan
ausencias estructurales, no datos perdidos. Ver clasificación detallada en la sección 4.

In [4]:
# Extraemos solo las columnas con nulos
nulos = df.isnull().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)

print(f'Total columnas con nulos: {len(nulos)}')
print(nulos)

Total columnas con nulos: 19
PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtExposure      38
BsmtFinType2      38
BsmtQual          37
BsmtCond          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
dtype: int64


## 4. Clasificación de Nulos por Tipo

Se define explícitamente si cada columna con nulos tiene una ausencia estructural o un dato real faltante.
Esta clasificación determina la estrategia de imputación: codificar vs. estimar.

**Nulos estructurales** → la propiedad realmente no tiene esa característica.
El valor `"NA"` en el CSV original de Kaggle lo indica explícitamente en el diccionario de datos.
Ejemplos: sin piscina → `PoolQC=NA`, sin garaje → `GarageType=NA`.

**Nulos reales** → información que debería estar registrada pero está ausente.
Solo 3 variables: `LotFrontage` (259), `MasVnrArea` (8) y `Electrical` (1).
Tratamiento diferenciado para cada una según volumen y distribución.

In [ ]:
# Clasificación explícita de cada columna con nulos
# 'estructural' → la ausencia es el dato real (la propiedad no tiene esa característica)
# 'real'        → el dato debería existir pero no fue registrado
clasificacion_nulos = {
    'PoolQC':       'estructural',  # NA = No tiene piscina (95% de las propiedades)
    'LotFrontage':  'real',         # Debería tener frente a calle — se imputa por vecindario
    'MiscFeature':  'estructural',  # NA = Sin características misceláneas (cancha, shed, etc.)
    'Alley':        'estructural',  # NA = Sin acceso por callejón
    'Fence':        'estructural',  # NA = Sin cerca
    'MasVnrType':   'estructural',  # None = Sin revestimiento de mampostería
    'FireplaceQu':  'estructural',  # NA = Sin chimenea (Fireplaces=0 confirma la ausencia)
    'GarageType':   'estructural',  # NA = Sin garaje (GarageCars=0 y GarageArea=0)
    'GarageYrBlt':  'estructural',  # NA = Sin garaje — se codifica como 0
    'GarageFinish': 'estructural',  # NA = Sin garaje
    'GarageQual':   'estructural',  # NA = Sin garaje
    'GarageCond':   'estructural',  # NA = Sin garaje
    'BsmtExposure': 'estructural',  # NA = Sin sótano
    'BsmtFinType2': 'estructural',  # NA = Sin sótano
    'BsmtQual':     'estructural',  # NA = Sin sótano (TotalBsmtSF=0 confirma)
    'BsmtCond':     'estructural',  # NA = Sin sótano
    'BsmtFinType1': 'estructural',  # NA = Sin sótano
    'MasVnrArea':   'real',         # Solo 8 nulos — se imputa con mediana global
    'Electrical':   'real',         # Solo 1 nulo — se imputa con moda
}

## 5. Tratamiento de Nulos Estructurales

Las 15 columnas categóricas estructurales se rellenan con `"None"` para representar la ausencia
como una categoría válida. `GarageYrBlt` se rellena con `0` (año inexistente → sin garaje).

Se asigna `"None"` como string para representar la ausencia como una **categoría válida**.
Esto permite que el OrdinalEncoder y OneHotEncoder del Notebook 03 procesen correctamente
la ausencia sin confundirla con un dato faltante.

`GarageYrBlt` (numérica) recibe `0` porque un año de construcción `0` señala inequívocamente
la inexistencia del garaje — no hay riesgo de confusión con un año real.

In [ ]:
# Lista de variables categóricas con nulos estructurales
# fillna('None') reemplaza NaN por el string "None" — representará la ausencia en el encoding
categoricas_estructurales = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'MasVnrType',
    'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual',
    'GarageCond', 'BsmtExposure', 'BsmtFinType2', 'BsmtQual',
    'BsmtCond', 'BsmtFinType1'
]

# Aplicamos el relleno a todas las categóricas estructurales en un solo loop
for col in categoricas_estructurales:
    df[col] = df[col].fillna('None')

# GarageYrBlt es numérica — se usa 0 para indicar que no hay garaje (año inexistente)
df['GarageYrBlt'] = df['GarageYrBlt'].fillna(0)

# Verificamos que no queden nulos en las columnas tratadas
nulos_post = df[categoricas_estructurales + ['GarageYrBlt']].isnull().sum()
print("Nulos restantes en variables estructurales:")
print(nulos_post[nulos_post > 0])

## 6. Tratamiento de Nulos Reales

Tres variables tienen nulos reales (información que debería existir):
- **`LotFrontage` (259 nulos):** se imputa con la mediana por vecindario — las propiedades de un mismo barrio tienden a tener frentes de lote similares
- **`MasVnrArea` (8 nulos):** se imputa con la mediana global — cantidad mínima no justifica agrupación  
- **`Electrical` (1 nulo):** se imputa con la moda global — categoría más frecuente es lo más probable

Estrategia de imputación para cada variable real:

1. **`LotFrontage`** → mediana agrupada por `Neighborhood`. Las propiedades en el mismo barrio
   tienen tamaños y diseños de lotes similares, haciendo esta agrupación más precisa que la mediana global.
2. **`MasVnrArea`** → mediana global. Solo 8 nulos no justifican el costo de una agrupación adicional;
   la mediana es robusta ante los valores cero (propiedades sin mampostería).
3. **`Electrical`** → moda global. Solo 1 nulo; la categoría más frecuente es la opción más razonable.

In [ ]:
# LotFrontage → mediana agrupada por Neighborhood
# groupby().transform('median') calcula la mediana por grupo y la asigna
# a cada fila según su vecindario — respeta la variación geográfica del tamaño del lote
df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median())
)

# MasVnrArea — solo 8 nulos: mediana global es suficientemente representativa
df['MasVnrArea'] = df['MasVnrArea'].fillna(df['MasVnrArea'].median())

# Electrical — solo 1 nulo: moda global (valor más frecuente en el dataset)
# mode()[0] extrae el primer valor si hay empate en la moda
df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0])

In [ ]:
# Verificación final: confirma que el dataset está completamente libre de nulos
# Si el resultado es 0, el dataset está listo para el Feature Engineering
total_nulos = df.isnull().sum().sum()
print(f'Total nulos restantes en el dataset: {total_nulos}')

## 7. Almacenamiento del Dataset Limpio

Se guarda el dataset tratado como `train_clean.csv` para ser consumido en el Notebook 03
(Feature Engineering). El archivo mantiene las 81 columnas originales con nulos resueltos.

Se guarda el DataFrame como `data/train_clean.csv` listo para ser consumido
en el Notebook 03. El archivo conserva las **81 columnas originales** con todos los nulos resueltos
y los valores estructurales representados como `"None"` o `0`.

In [ ]:
# Guardamos el dataset limpio en data/ al nivel raíz del proyecto
# parents=True crea directorios intermedios si no existen; exist_ok=True evita error si ya existe
clean_path = Path('../../data/train_clean.csv')
clean_path.parent.mkdir(parents=True, exist_ok=True)

# index=False evita guardar el índice de filas como columna extra
df.to_csv(clean_path, index=False)

print(f'Dataset limpio guardado en: {clean_path.resolve()}')
print(f'Filas: {df.shape[0]}')
print(f'Columnas: {df.shape[1]}')
print(f'Nulos totales: {df.isnull().sum().sum()}')

---

## Conclusiones — Tratamiento de Missing Values

### Resumen del tratamiento

| Tipo | Variables | Estrategia | Resultado |
|------|-----------|------------|-----------|
| Estructurales categóricas | 15 | `fillna("None")` | Ausencia como categoría válida |
| Estructurales numéricas | 1 (`GarageYrBlt`) | `fillna(0)` | Año=0 → sin garaje |
| Reales por volumen medio | 1 (`LotFrontage`) | Mediana por Neighborhood | Respeta variación geográfica |
| Reales por bajo volumen | 2 (`MasVnrArea`, `Electrical`) | Mediana/moda global | Suficiente con n < 10 |

### Decisiones clave y su justificación
- **No se eliminó ninguna fila:** las 1,460 observaciones se conservan íntegras
- **No se imputaron nulos estructurales:** hacerlo introduciría información falsa (p.ej. asignar `"TA"` de calidad a un sótano que no existe)
- **`"None"` como string:** permite al OrdinalEncoder del Notebook 03 procesar la ausencia como categoría de la jerarquía más baja (posición 0)
- **Mediana sobre media:** robusta ante los outliers de precio que caracterizan este dataset

### Resultado final
Dataset `train_clean.csv` con **1,460 filas × 81 columnas** y **0 valores nulos**.

### Próximo paso
**Notebook 03** — Encoding de variables categóricas y creación de features temporales.